<a href="https://colab.research.google.com/github/kitlapp/NLP-Course/blob/main/pr_043_RAG_GradioUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Section 4.3 UI Development with Gradio & Deployment on Hugging Face Spaces

This Notebook implements a Retrieval-Augmented Generation (RAG) Medical Drug Review Assistant chatbot interface using Gradio.   

**RAG Retrieval Engine:**  
Through a search function (get_context) top most relevant patient reviews based on metrics like side effects, satisfaction, effectiveness, and ease of use are retrieved. This is done following 4.2 Task.  

**Web UI Interface:**
Builds and launches an interactive Gradio web application featuring a customized header, model parameter controls (sliders for similarity count and generation creativity), and a context inspector panel.

In [24]:
#Vector Database to use instead of Cosine Similarity
!pip install faiss-cpu

#Import Libraries

In [25]:
# Standard library
from pathlib import Path
import os
from dotenv import load_dotenv
# Third-party libraries
import pandas as pd
import numpy as np
#
import gradio as gr
from openai import OpenAI
#
import io
import IPython.display
from PIL import Image
import base64
import requests, json
#For RAG
from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, TextStreamer, GenerationConfig
from transformers import pipeline
#
import faiss #Vector Database
#from huggingface_hub.inference._generated.types import question_answering
#
pd.set_option('display.max_colwidth', None)


#Define Paths and Environmental Variables

In [26]:
# from google.colab import drive
# drive.mount('/content/drive')

In [27]:
# os.chdir('/content/drive/MyDrive/MSc NLP/NaturalLanguageProcessing-main/NaturalLanguageProcessing-main')
# # Verify the move
# print("Current directory:", os.getcwd())

In [28]:
# load_dotenv()

In [29]:
# Path to root shared data folder (should work for every user after adding the shortcut)
ROOT_DATA_DIR = Path("/content/drive/MyDrive/nlp_project_data")

# Check that the folder exists
if ROOT_DATA_DIR.exists():
    print(f"Shared dataset folder found: {ROOT_DATA_DIR}")
else:
    raise FileNotFoundError(
        f"Dataset folder not found: {ROOT_DATA_DIR}\n"
        "Read the README instructions or the text cells in the sections "
        "'Mount Google Drive' and 'Handle Paths'.\n"
        "Ensure that the shared dataset folder has been added as a shortcut "
        "inside your My Drive, then rerun this cell."
    )

Shared dataset folder found: /content/drive/MyDrive/nlp_project_data


In [30]:
# Define the path to the saved CSV file
csv_path = ROOT_DATA_DIR / "df_rag_block.csv"

# Read the CSV back into a pandas DataFrame
df_rag = pd.read_csv(csv_path)

print(f"DataFrame successfully loaded! Shape: {df_rag.shape}")

DataFrame successfully loaded! Shape: (319933, 13)


#Define CPU or GPU

In [31]:
if torch.cuda.device_count()>0:
    my_device = "cuda"
    print(f"You have {torch.cuda.device_count()} GPUs available.")
else:
    my_device = "cpu"
    print("You have no GPUs available. Running on CPU.")

You have no GPUs available. Running on CPU.


# Encoder Model

The emeddings model will be needed to create the embeddings of the query of the user

In [32]:
embeddings_model_name = 'BAAI/bge-small-en-v1.5'

In [23]:
embeddings_model_name = 'BAAI/bge-small-en-v1.5'
#embeddings_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
#Note: Because the model files are already in ./HF-CACHE, Hugging Face's library will automatically detect them locally and load them instantly without hitting the internet in a second run
embeddings_model = SentenceTransformer(embeddings_model_name,
                                       cache_folder="./HF-CACHE",
                                       device=my_device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [33]:
# #REad the embeddings model
# # Load the model directly from your local folder
# safe_model_name = embeddings_model_name.replace("/", "_")
# file_name_embeddingsMODEL = f"review_embeddingsMODEL_{safe_model_name}"
# embeddings_model = SentenceTransformer(file_name_embeddingsMODEL, device=my_device)

#Read the Vector Database

In [34]:
#Load the indexes from the vector database

# Define the exact path where it was saved
safe_model_name = embeddings_model_name.replace("/", "_")
file_name_index = f"review_index_{safe_model_name}.faiss"
index_path = ROOT_DATA_DIR / file_name_index

# Check if the file exists before trying to load (good practice!)
if index_path.exists():
    # Read the index
    # Note: read_index requires a string, not a Path object
    index = faiss.read_index(str(index_path))
    print(f"Successfully loaded index with {index.ntotal} vectors.")
else:
    print(f"Error: The file {index_path} was not found. Check your file name or path.")

Successfully loaded index with 319933 vectors.


#
Decoder Model

In [35]:
llm_model_name = "Qwen/Qwen3.5-0.8B"
tokenizer = AutoTokenizer.from_pretrained(llm_model_name, cache_dir="./HF-CACHE")
llm_model = AutoModelForCausalLM.from_pretrained(   llm_model_name,
                                                cache_dir="./HF-CACHE",
                                                device_map="auto",
                                                torch_dtype="auto")
#The pad_token is just a way to make all your data the same size so your GPU can process it efficiently.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Pad token was None, so it was set to eos token.")
    #As soon as the model predicts a single "token" (a word or part of a word), the streamer catches it, decodes it using the tokenizer you provided, and prints it to your output immediately.

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

#Setting Up the Chatbot

##Top K most relevant documents to query

Parameters that affect the bot behavior

In [36]:
#Parameters
#most relevand rag_blocks with the query
k_mostrelevant = 30
#Length of the output
max_new_tokens=2048
#How cretive the bot will be. 0.1 - 0.3 a rigid factual bot, while 0.8 to 1 more creative
temperature = 0.7
#how much of the history the bot remembers
max_convo_length=10

In [37]:
#--------------------------------------------------------------------------------
#Create embeddings of the question adn retrieve the top k most relevant
#documents to the user question from the vector database
#

def get_context(query, embeddings_model, index, df, k=k_mostrelevant):
    """
    Retrieves the top k most relevant documents from the vector store.

    Args:
        query (str): The user's question.
        embeddings_model: Your loaded SentenceTransformer model.
        index: Your FAISS index object.
        df (pd.DataFrame): Your main dataset (df_rag).
        k (int): Number of chunks to retrieve.

    Returns:
        list: A list of text chunks (rag_block) containing the context.
    """
    # Embed the user query
    # The model expects a list, so we wrap the querymax_convo_length in []
    query_vector = embeddings_model.encode([query])

    # FAISS strictly requires float32 data types
    query_vector = np.array(query_vector).astype('float32')

    # Search the index
    # distances: how similar the result is (lower is better for L2 distance)
    # indices: the row numbers in your dataframe
    distances, indices = index.search(query_vector, k)

    # Retrieve the context
    # indices[0] is the list of results for our 1 query
    # We use .iloc to extract these specific rows from the dataframe
    #retrieved_texts = df_rag.iloc[indices[0]]['rag_block'].tolist()
    retrieved_texts = df_rag.iloc[indices[0]]['rag_block'].unique().tolist() # added the unique since many times the same reviews are listed in the tests!

    return retrieved_texts

##Functions to format Request and Response

In [38]:
# Set up the local Hugging Face generation pipeline using your loaded model & tokenizer
qwen_pipeline = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer
)

#  Define your custom chat function so it accepts the parameters Gradio sends
def chat(system_prompt, user_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    # Put max_new_tokens right here inside the GenerationConfig
    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,  # <--- HERE
        max_length=None,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    # Generate text locally using Qwen
    outputs = qwen_pipeline(
        messages,
        generation_config=gen_config)

    # Safely extract and return just the assistant's response text
    return outputs[0]["generated_text"][-1]["content"]

#  Helper to format past conversation history
def format_chat_prompt(message, chat_history, max_convo_length):
    prompt = ""
    for turn in chat_history[-max_convo_length:]:
        role = turn.get("role")
        content = turn.get("content")
        speaker = "User" if role == "user" else "Assistant"
        prompt = f"{prompt}\n{speaker}: {content}"

    prompt = f"{prompt}\nUser: {message}\nAssistant:"
    return prompt

# 4. Main response function handling RAG retrieval + Qwen generation
def respond(message, chat_history, k_val, temp_val, max_convo_length=max_convo_length):
    # Retrieve top relevant reviews from your FAISS index
    context_list = get_context(message, embeddings_model, index, df_rag, k=int(k_val))
    context_list = list(dict.fromkeys(context_list)) #I was getting too many similar outcomes in the BOT
    context_str = "\n\n".join(context_list)

        # Format the user's latest query + past conversation history
    formatted_prompt = format_chat_prompt(message, chat_history, max_convo_length)

    # Your custom medical assistant system instruction
    system_instruction = (
        "You are a medical research assistant. "
        "Use the provided <context> (which includes patient reviews and ratings such as EaseofUse, Effectiveness and Satisfaction) to answer the <question>. "
        "Prioritize answers with larger UsefulCount. "
        "Do not make up medical facts outside of the provided context. "
        "Do not use outside knowledge.\n\n"
        "Instructions:\n"
        "- If the question asks about 'side effects', prioritize the 'Side Effects' field.\n"
        "- If the question asks about 'effectiveness' or comparative drug performance, prioritize the numerical 'Effectiveness' ratings over subjective text sentiment.\n"
        "- If the question asks about 'satisfaction', prioritize the numerical 'Satisfaction' ratings over subjective text sentiment.\n"
        "- If the question asks about 'ease of use', prioritize the numerical 'EaseofUse' ratings and specific usability comments in the reviews.\n"
        "- When performing drug comparisons, prioritize numerical ratings across the metadata rather than selective review quotes.\n"
        "- When summarizing data or trends from the snippets, synthesize the patterns directly from the retrieved text rather than refusing due to a lack of global data.\n"
        "- Do not invent rating scales (like 6-10) if the ratings are out of 5.\n"
        "- When answering, directly address the user's prompt using the patterns, counts, and examples found within the retrieved text snippets.\n"
        "- If a specific data point is missing from the snippets, concisely state what the retrieved snippets show rather than issuing a blanket refusal.\n"
        "- Answer directly and concisely based on the text provided.\n"
        "-If a comparative question is asked (e.g., comparing age groups) and the retrieved context shows identical standardized metadata for both groups, explicitly state that the context shows no difference rather than forcing a distinction."
    )

    # Combine context and prompt payload
    user_prompt_with_rag = f"Context:\n{context_str}\n\nConversation/Question:\n{formatted_prompt}"

    # Call local Qwen chat generation
    bot_message = chat(
        system_prompt=system_instruction,
        user_prompt=user_prompt_with_rag
    )

    # Append turns using the modern dictionary format
    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": bot_message})

    return "", chat_history, context_str


#Launch GRADIO UI

In [39]:
# # Launch Gradio UI
# with gr.Blocks() as demo:
#     gr.Markdown("# 💊 Medical Drug Review RAG Assistant (Powered by Qwen)")

#     chatbot = gr.Chatbot(height=300)
#     msg = gr.Textbox(label="Ask a medical question (e.g., What are the side effects of Lisinopril?)")

#     with gr.Row():
#         btn = gr.Button("Submit")
#         clear = gr.ClearButton(components=[msg, chatbot], value="Clear Console")

#     btn.click(respond, inputs=[msg, chatbot], outputs=[msg, chatbot])
#     msg.submit(respond, inputs=[msg, chatbot], outputs=[msg, chatbot])

# gr.close_all()
# demo.launch(debug=True)

#Formatted Alternative of Gardio UI

In [41]:
# Custom CSS for the shadow separator line, WebMD full-length header box, and perfectly matching default Gradio theme purple sliders
custom_css = """
.webmd-header-box {
    background-color: #0070AF;
    padding: 20px 25px;
    border-radius: 10px;
    color: white;
    box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
    margin-bottom: 20px;
    width: 100%;
}
.webmd-header-box h1, .webmd-header-box p, .webmd-header-box * {
    color: white !important;
}
.shadow-divider {
    border: none;
    height: 4px;
    background: linear-gradient(to bottom, rgba(0,0,0,0.1), rgba(0,0,0,0.02));
    box-shadow: inset 0 2px 4px rgba(0,0,0,0.06);
    margin: 15px 0;
}
/* Style the primary submit button to use a lighter WebMD-themed blue tint */
button.primary {
    background-color: #3B95D1 !important;
    border-color: #3B95D1 !important;
    color: white !important;
}
button.primary:hover {
    background-color: #2E81B8 !important;
}

/* Match both the slider track accent and thumb precisely to Gradio's Soft/Indigo theme purple (#6366f1) */
input[type="range"] {
    accent-color: #6366f1 !important;
}
input[type="range"]::-webkit-slider-thumb {
    background-color: #6366f1 !important;
    border: 2px solid #6366f1 !important;
}
input[type="range"]::-moz-range-thumb {
    background-color: #6366f1 !important;
    border: 2px solid #6366f1 !important;
}
input[type="range"]::-ms-thumb {
    background-color: #6366f1 !important;
    border: 2px solid #6366f1 !important;
}
"""

# Apply a sleek built-in Gradio theme (e.g., Soft or Glass)
custom_theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="blue",
).set(
    button_primary_background_fill="*primary_500",
    button_primary_background_fill_hover="*primary_600",
)

with gr.Blocks(theme=custom_theme, css=custom_css) as demo:
    # Header Section inside a full-length WebMD-themed blue box with white text
    with gr.Row():
        gr.Markdown(
            """
            <div class="webmd-header-box">
                <h1 style="margin: 0; font-size: 26px;">💊 Medical Drug Review Assistant</h1>
                <p style="margin: 8px 0 0 0; font-size: 15px; opacity: 0.95;">
                    &nbsp;&nbsp;&nbsp;&nbsp;<em>An NLP application for analyzing WebMD patient reviews</em>
                </p>
            </div>
            """
        )

    with gr.Row():
        # Left Column: Chatbot Interface (Main focus)
        with gr.Column(scale=3):
            # show_label=False removes the "Conversation" box in the top-left corner
            chatbot = gr.Chatbot(height=400, show_label=False)

            # A nice separator line with a shadow effect
            gr.HTML('<div class="shadow-divider"></div>')

            msg = gr.Textbox(
                label="Query",
                placeholder="Ask a medical question (e.g., What are the side effects of Lisinopril?)...",
                container=True
            )

            # Stacked Buttons: Submit on top, Clear right below it in a vertical column
            with gr.Column():
                btn = gr.Button("Submit", variant="primary")
                clear = gr.ClearButton(components=[msg, chatbot], value="Clear Conversation")

            # Clickable Example Pills for Quick Testing
            gr.Examples(
                examples=[
                    "What are the side effects of Lisinopril?",
                    "Do older patients (75+) report different side effects than younger ones?",
                    "What do reviews say about its effectiveness?"
                ],
                inputs=msg
            )

        # Right Column: Inspector / Settings Panel (Great for presentations!)
        with gr.Column(scale=1):
            gr.Markdown("## 📊 Project Dashboard")
            with gr.Accordion("⚙️ Model Controls", open=True):
                slider_k = gr.Slider(minimum=5, maximum=40, value=30, step=5, label="Most Similar Reviews")
                slider_temp = gr.Slider(minimum=0.1, maximum=1.0, value=0.7, step=0.1, label="Answering Creativity")

            with gr.Accordion("🔍 Retrieved Context Inspector", open=False):
                context_output = gr.Textbox(label="Most Relevant Documents Extracted", lines=10, interactive=False)

    # Wire up events (Outputs both the updated chat and the context inspector box)
    btn.click(
        respond,
        inputs=[msg, chatbot, slider_k, slider_temp],
        outputs=[msg, chatbot, context_output]
    )
    msg.submit(
        respond,
        inputs=[msg, chatbot, slider_k, slider_temp],
        outputs=[msg, chatbot, context_output]
    )

gr.close_all()
demo.launch(debug=True)

/tmp/ipykernel_1521/3169919940.py:59: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=custom_theme, css=custom_css) as demo:


Closing server running on port: 7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://68b32809c6ab505eb8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://68b32809c6ab505eb8.gradio.live
